# 10 · Complete irrigation technology shares

Replay original Cell 113: complete valid source shares by nearest valid same-country year (average ties), followed by >=2 approved same-year regional peers. The frozen 14-country peer definitions are preserved in `config/technology-peers.csv`. Compare all three shares with the approved table before claiming parity.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

COLS = ['surface_share','sprinkler_share','drip_share']
KEYS = ['country_iso3','year']


def prepare_source(source: pd.DataFrame) -> pd.DataFrame:
    source = source.copy()
    if 'country_iso3' not in source and 'iso3' in source:
        source = source.rename(columns={'iso3':'country_iso3'})
    missing = set(KEYS+COLS)-set(source)
    if missing:raise ValueError(f'Missing source columns: {sorted(missing)}')
    source['country_iso3'] = source.country_iso3.astype(str).str.strip().str.upper()
    source['year'] = pd.to_numeric(source.year,errors='raise').astype(int)
    if source.duplicated(KEYS).any():raise ValueError('Non-unique source country-year keys')
    for col in COLS:source[col] = pd.to_numeric(source[col],errors='coerce')
    vals = source[COLS].to_numpy(float)
    source['is_complete'] = (np.isfinite(vals).all(axis=1) &
        (vals>=0).all(axis=1) & (vals<=1).all(axis=1) &
        np.isclose(vals.sum(axis=1),1.0,atol=0.005,rtol=0))
    return source


def complete(required: pd.DataFrame, source: pd.DataFrame, peers: pd.DataFrame,
             min_peers: int = 2) -> pd.DataFrame:
    missing = set(KEYS)-set(required)
    if missing:raise ValueError(f'Missing required keys: {sorted(missing)}')
    base = required[KEYS].drop_duplicates().copy()
    base['year'] = pd.to_numeric(base.year,errors='raise').astype(int)
    base['country_iso3'] = base.country_iso3.astype(str).str.strip().str.upper()
    if base[KEYS].isna().any().any():raise ValueError('Null required country-year key')
    source=prepare_source(source)
    if peers.duplicated('target_country_iso3').any():raise ValueError('Duplicate frozen peer assignments')
    peer_lookup=peers.set_index('target_country_iso3')
    complete_source=source[source.is_complete]
    results=[]
    for r in base.sort_values(KEYS).itertuples(index=False):
        iso,year=r.country_iso3,int(r.year)
        same=complete_source[(complete_source.country_iso3==iso)&(complete_source.year==year)]
        route='SOURCE_COMPLETE'
        confidence='SOURCE_RECORDED'
        donor_years=str(year)
        donors=iso
        if not same.empty:
            vec=same.iloc[0][COLS].to_numpy(float)
        else:
            alt=complete_source[complete_source.country_iso3==iso].copy()
            if not alt.empty:
                delta=(alt.year-year).abs()
                tied=alt[delta==delta.min()]
                vec=tied[COLS].mean(axis=0).to_numpy(float)
                donor_years=';'.join(map(str,sorted(tied.year.tolist())))
                route='R1_SAME_COUNTRY_NEAREST_VALID_YEAR'
                confidence='LOW'
            else:
                if iso not in peer_lookup.index:
                    raise ValueError(f'No approved regional peer definition for unresolved {iso} {year}')
                peers_for_country=str(peer_lookup.loc[iso,'peer_countries']).split(';')
                relevant=complete_source[(complete_source.year==year)&
                    complete_source.country_iso3.isin(peers_for_country)]
                if relevant.country_iso3.nunique()<min_peers:
                    raise ValueError(f'Insufficient approved complete same-year peers for {iso} {year}')
                vec=relevant[COLS].mean(axis=0).to_numpy(float)
                donors=';'.join(sorted(relevant.country_iso3.tolist()))
                donor_years=str(year)
                route='R2_SAME_YEAR_REGIONAL_PEER_MEAN'
                confidence='VERY_LOW'
        if not(np.isfinite(vec).all() and (vec>=-1e-10).all() and
               np.isclose(vec.sum(),1.0,atol=0.005,rtol=0)):
            raise ValueError(f'Invalid completed composition for {iso} {year}: {vec}')
        results.append(dict(zip(COLS,vec))|{'country_iso3':iso,'year':year,
            'technology_resolution_route':route,'technology_completion_confidence':confidence,
            'technology_donor_countries':donors,'technology_donor_years':donor_years})
    return pd.DataFrame(results).sort_values(KEYS).reset_index(drop=True)


def compare_reference(rebuilt:pd.DataFrame,approved:pd.DataFrame)->dict:
    approved=prepare_source(approved)
    merged=rebuilt.merge(approved,on=KEYS,how='outer',suffixes=('_new','_accepted'),indicator=True)
    bycol={}
    for col in COLS:
        both=merged['_merge'].eq('both').to_numpy()
        a=pd.to_numeric(merged[f'{col}_new'],errors='coerce').to_numpy(float)
        b=pd.to_numeric(merged[f'{col}_accepted'],errors='coerce').to_numpy(float)
        bycol[col]=int(np.count_nonzero(both & ~np.isclose(a,b,atol=1e-9,rtol=1e-9,equal_nan=True)))
    result={'rebuilt_rows':len(rebuilt),'approved_rows':len(approved),
            'missing_or_extra_keys':int(merged['_merge'].ne('both').sum()),
            'mismatches':bycol}
    result['status']='MATCH' if result['missing_or_extra_keys']==0 and not any(bycol.values()) else 'MISMATCH'
    return result

In [ ]:
NET=DATA/'crop'/'01_FABLE_crop_water_requirement_master_2000_2020_FINAL.csv.gz'
if not NET.is_file():NET=NET.with_suffix('')
TECH=DATA/'technology'/'03_FABLE_global_irrigation_system_shares_2000_2020.csv'
PEERS=ROOT/'config'/'technology-peers.csv'
APPROVED=DATA/'technology'/'12_FABLE_irrigation_technology_shares_2000_2020_FINAL.csv'
if not NET.is_file():raise FileNotFoundError(NET)
if not TECH.is_file():raise FileNotFoundError(TECH)
required=pd.read_csv(NET,usecols=KEYS)
completed=complete(required,pd.read_csv(TECH),pd.read_csv(PEERS))
assert len(completed)==824, 'Unexpected FABLE-required country-year universe'
print(completed.technology_resolution_route.value_counts())
if APPROVED.is_file():
    print('Accepted reference check:',compare_reference(completed,pd.read_csv(APPROVED)))
else:print('Approved 824-row reference absent; numerical parity not tested')

In [ ]:
OUT=RUNS/'technology-completion'/'completed-shares.csv'
OUT.parent.mkdir(parents=True,exist_ok=True)
if OUT.exists():raise FileExistsError(OUT)
completed.to_csv(OUT,index=False)